### Interactive Tutorial: Morpho-Spectral Projection Analysis (MSPA)
#### This notebook demonstrates the use of the `MSPA` operator on synthetic leaf reflectance spectra ($401\text{--}2400\text{ nm}$, $D = 1,580$ bands), comparing it with `PCA` and `PLS-DA` without the need for external data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# If running in Google Colab after uploading to GitHub, uncomment the line below:
# !pip install -q git+https://github.com/Lucas-Osco/mspa.git
from mspa import MSPA, generate_synthetic_vegetation_spectra

### 1. Visualization of Synthetic Spectral Signatures and Permutation Invariance

In [ ]:
# Generate a physically realistic synthetic dataset (N = 450 samples, D = 1580 bands, C = 3 classes)
X_df, y, wavelengths = generate_synthetic_vegetation_spectra(n_samples=450, random_state=42)
class_names = {
    0: "Class 0: Healthy Control",
    1: "Class 1: Early Chlorosis (Red-Edge Blue-Shift)",
    2: "Class 2: Vascular Stress (SWIR Concavity Shift)"
}
print(f"Dataset shape: {X_df.shape} | Classes: {np.bincount(y)}")

In [ ]:
# Column Permutation Experiment:
rng = np.random.default_rng(777)
perm_indices = rng.permutation(X_df.shape[1])
X_shuffled = X_df.to_numpy()[:, perm_indices]

Z_pca_orig = PCA(n_components=10, random_state=42).fit_transform(X_df.to_numpy())
Z_pca_shuf = PCA(n_components=10, random_state=42).fit_transform(X_shuffled)
diff_pca = np.max(np.abs(Z_pca_orig - Z_pca_shuf))

Z_mspa_orig = MSPA(n_components=64, wavelengths=wavelengths).fit_transform(X_df.to_numpy(), y)
Z_mspa_shuf = MSPA(n_components=64, wavelengths=wavelengths).fit_transform(X_shuffled, y)
diff_mspa = np.mean(np.abs(Z_mspa_orig - Z_mspa_shuf))

print(f"Maximum difference in PCA scores after shuffling the 1,580 wavelengths: {diff_pca:.2e} (Invariant / Order-agnostic)")
print(f"Mean difference in MSPA scores after shuffling the 1,580 wavelengths: {diff_mspa:.4f} (Sensitive to contiguous morphology)")

### 2. Cross-Validation Benchmark (5-Fold CV): Full Spectrum vs. PCA vs. PLS-DA vs. MSPA

In [ ]:
# %%
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_rf = {"0_RAW (1580)": [], "1_PCA (50)": [], "2_PLSDA (15)": [], "5_MSPA (64)": []}
results_svm = {"0_RAW (1580)": [], "1_PCA (50)": [], "2_PLSDA (15)": [], "5_MSPA (64)": []}

for tr_idx, te_idx in skf.split(X_df, y):
    X_tr, X_te = X_df.iloc[tr_idx], X_df.iloc[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)

    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    svm = SVC(C=1.0, kernel="rbf", random_state=42)

    # 1. 0_RAW (1580 bandas)
    rf.fit(X_tr_s, y_tr); results_rf["0_RAW (1580)"].append(f1_score(y_te, rf.predict(X_te_s), average="macro"))
    svm.fit(X_tr_s, y_tr); results_svm["0_RAW (1580)"].append(f1_score(y_te, svm.predict(X_te_s), average="macro"))

    # 2. 1_PCA (50 componentes)
    pca = PCA(n_components=50, random_state=42).fit(X_tr_s)
    rf.fit(pca.transform(X_tr_s), y_tr); results_rf["1_PCA (50)"].append(f1_score(y_te, rf.predict(pca.transform(X_te_s)), average="macro"))
    svm.fit(pca.transform(X_tr_s), y_tr); results_svm["1_PCA (50)"].append(f1_score(y_te, svm.predict(pca.transform(X_te_s)), average="macro"))

    # 3. 2_PLSDA (15 variáveis latentes)
    Y_ohe = pd.get_dummies(y_tr).values
    pls = PLSRegression(n_components=15, scale=False).fit(X_tr_s, Y_ohe)
    rf.fit(pls.transform(X_tr_s), y_tr); results_rf["2_PLSDA (15)"].append(f1_score(y_te, rf.predict(pls.transform(X_te_s)), average="macro"))
    svm.fit(pls.transform(X_tr_s), y_tr); results_svm["2_PLSDA (15)"].append(f1_score(y_te, svm.predict(pls.transform(X_te_s)), average="macro"))

    # 4. 5_MSPA (64 variáveis morfo-espectrais)
    mspa = MSPA(n_components=64).fit(X_tr, y_tr)
    Z_tr_m, Z_te_m = mspa.transform(X_tr), mspa.transform(X_te)
    rf.fit(Z_tr_m, y_tr); results_rf["5_MSPA (64)"].append(f1_score(y_te, rf.predict(Z_te_m), average="macro"))
    svm.fit(Z_tr_m, y_tr); results_svm["5_MSPA (64)"].append(f1_score(y_te, svm.predict(Z_te_m), average="macro"))

summary_df = pd.DataFrame({
    "Representation": list(results_rf.keys()),
    "RandomForest_F1_Mean": [np.mean(v) for v in results_rf.values()],
    "RandomForest_F1_Std": [np.std(v) for v in results_rf.values()],
    "SVM_RBF_F1_Mean": [np.mean(v) for v in results_svm.values()],
    "SVM_RBF_F1_Std": [np.std(v) for v in results_svm.values()],
})
print("\n=== Synthetic Benchmark Result (5-Fold Cross-Validation) ===")
print(summary_df.round(4).to_string(index=False))

### 3. Comparative Chart (Synthetic Curves + F1-Macro Performance)

In [ ]:
# %%
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.8), dpi=150)

colors = ["#2C3E50", "#A85042", "#6E7F5B"]
b_masks = [(wavelengths <= 1330), (wavelengths >= 1491) & (wavelengths <= 1770), (wavelengths >= 2031)]

for c_id, c_label in class_names.items():
    mean_spec = X_df.to_numpy()[y == c_id].mean(axis=0)
    for b_i, bm in enumerate(b_masks):
        ax1.plot(wavelengths[bm], mean_spec[bm], color=colors[c_id], lw=1.6, label=c_label if b_i == 0 else None)

ax1.axvspan(1330, 1490, color="#E2E5E8", alpha=0.8)
ax1.axvspan(1770, 2030, color="#E2E5E8", alpha=0.8)
ax1.set_title("A. Synthetic Leaf Spectra (SNV, D = 1,580 Bands)", fontweight="bold")
ax1.set_xlabel("Wavelength (nm)")
ax1.set_ylabel("SNV Reflectance (Unitless)")
ax1.legend(loc="upper right", fontsize=8)
ax1.grid(True, linestyle="--", alpha=0.4)

x_pos = np.arange(len(summary_df))
w_bar = 0.35
ax2.bar(x_pos - w_bar/2, summary_df["RandomForest_F1_Mean"], width=w_bar, yerr=summary_df["RandomForest_F1_Std"], capsize=4, color="#3B5B6D", edgecolor="#1C2833", label="Random Forest")
ax2.bar(x_pos + w_bar/2, summary_df["SVM_RBF_F1_Mean"], width=w_bar, yerr=summary_df["SVM_RBF_F1_Std"], capsize=4, color="#6E7F5B", edgecolor="#1C2833", label="SVM (RBF Kernel)")
ax2.set_xticks(x_pos)
ax2.set_xticklabels(summary_df["Representation"])
ax2.set_ylim(0.70, 1.04)
ax2.set_title("B. 5-Fold CV Macro-F1 Across Representations", fontweight="bold")
ax2.set_ylabel("Macro F1-Score")
ax2.legend(loc="lower right")
ax2.grid(True, axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()